<a href="https://colab.research.google.com/github/kameshcodes/pydantic-tutorial/blob/main/basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$$\text{Pydantic Tutorial}$$

---

## **Table of Contents**

- [1. Introduction](#sec-1)
- [2. Field Validator](#sec-2)
  - [2.1 Type Validator](#sec-2-1)
    - [2.1.1 Complex Types](#complex-types)
    - [2.1.2 Optional Types](#optional-types)
    - [2.1.3 Defaults](#defaults)
  - [2.2 Data Validator](#sec-2-2)
    - [2.2.1 Built-in Validators](#builtin-validators)
    - [2.2.2 Field()](#field-constraints)
    - [2.2.3 Annotated](#annotated)
- [3. Field Validator (Custom Logic)](#sec-3)
  - [3.1 Validation](#sec-3-1)
  - [3.2 Transformation](#sec-3-2)
  - [3.3 Before and After Modes](#sec-3-3)
- [4. Model Validator](#sec-4)
  - [4.1 Validation](#sec-4-1)
- [5. Computed Field](#sec-5)
  - [3.1 Without Computed Field](#sec-5-1)
  - [3.2 With Computed Field](#sec-5-2)
- [6. Nested Models](#sec-6)
  - [4.1 The Problem with a Flat String](#sec-6-1)
  - [4.2 Using a Nested Model](#sec-6-2)
- [7. Serialization](#sec-7)
  - [5.1 model_dump()](#sec-7-1)
  - [5.2 model_dump_json()](#sec-7-2)
  - [5.3 include and exclude](#sec-7-3)

<a id='sec-1'></a>

## **1. Introduction**

---

Pydantic is a Python library that does two things:

- Checks that your data is the right **type**
- Checks that your data follows the right **rules** (range, format, length)

<br>

Here's why you need it. Python is **dynamically typed**. It doesn't check types at all. You can pass a string where an int is expected and Python won't complain. The bug shows up later, somewhere else, and by then it's much harder to debug.

#### **1.1 The Problem**

---

Let's start with a plain Python function with no validation at all:

In [ ]:
def insert_patient_data(name, age):
    print(f"Inserting patient data: Name={name}, Age={age}")
    print("Patient data inserted successfully.")

In [ ]:
# Valid input — works fine
insert_patient_data("John Doe", 30)

Inserting patient data: Name=John Doe, Age=30
Patient data inserted successfully.


In [ ]:
# Invalid input — age is a string, but Python doesn't care
insert_patient_data("Jane Smith", "Thirty")

Inserting patient data: Name=Jane Smith, Age=Thirty
Patient data inserted successfully.


**No error. No warning.** The bad data silently went through. In a real system, this kind of bug is hard to catch and expensive to fix.

#### **1.2 The if/else Approach and Why It Doesn't Scale**

---

The obvious fix is to add manual checks with `isinstance` and `if/else`:

In [ ]:
def insert_patient_data(name, age):
    if not isinstance(name, str):
        raise TypeError("name must be a string")
    if not isinstance(age, int):
        raise TypeError("age must be an integer")
    if age < 0:
        raise ValueError("age cannot be negative")

    print(f"Inserting patient data: Name={name}, Age={age}")
    print("Patient data inserted successfully.")

In [ ]:
try:
    insert_patient_data("Jane Smith", "Thirty")  # TypeError
except TypeError as e:
    print(e)

age must be an integer


In [ ]:
try:
    insert_patient_data("Jane Smith", -5)  # ValueError
except ValueError as e:
    print(e)

age cannot be negative


This patch is good for 2-3 fields. Once your schema grows, this approach breaks down. Add 10 fields and you're writing 30+ manual checks:

- No standard error format. You write your own message each time.
- Easy to miss a check when adding a new field.
- Hard to maintain across multiple functions.

**Pydantic replaces all of this with a model definition.**

<a id='sec-2'></a>

## **2. Field Validator**

---

Pydantic validates a field in two stages:

- **Type Validator**: is the value the right *type*? (`str`, `int`, `bool`, `List`, ...)
- **Data Validator**: does the value follow the *rules*? (correct format, within range, right length)

Both run automatically when you create a model instance. You don't have to call anything.

In [ ]:
!pip install pydantic -q

zsh:1: command not found: pip


<a id='sec-2-1'></a>

### **2.1 Type Validator**

---

This checks that each field has the correct type. If the value is close enough, Pydantic will **coerce** it for you. If it can't coerce, you get a `ValidationError` with a clear message.

3 steps to use it:

#### **Step 1: Define your model**

---

Subclass `BaseModel` and declare your fields with type hints. That's your contract. Pydantic enforces it on every object you create.

In [ ]:
from pydantic import BaseModel, ValidationError

class Patient(BaseModel):
    name: str
    age: int

#### **Step 2: Create an instance**

---

Pass your data in. Pydantic validates it on the spot. Let's try three scenarios:

<br>

**Case 1: Data is already the right type:**

In [ ]:
patient_info = {"name": "John Doe", "age": 30}  # age is already int
patient = Patient(**patient_info)
print(patient)

name='John Doe' age=30


**Case 2: Pydantic coerces the value:**

In [ ]:
patient_info = {"name": "John Doe", "age": "30"}  # "30" → coerced to int 30
patient = Patient(**patient_info)
print(patient)

name='John Doe' age=30


**Case 3: Can't coerce, raises ValidationError:**

In [ ]:
try:
    patient_info = {"name": "John Doe", "age": "Thirty"}  # can't convert
    patient = Patient(**patient_info)
except ValidationError as e:
    print(e)

1 validation error for Patient
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


Notice how precise the error is. It tells you the **field name**, **what went wrong**, and **what value was passed**. No manual error message needed.

#### **Step 3: Use the validated object**

---

Once it's created, the data is guaranteed clean. Your function just uses it. No checks needed inside.

<a id='complex-types'></a>

#### **2.1.1 Complex Types**

---

So far we've used `str` and `int`. Pydantic handles complex types just as easily. It validates *inside* collections too, not just the collection itself.

In [ ]:
from pydantic import BaseModel
from typing import List, Dict

class Patient(BaseModel):
    name: str
    age: int
    married: bool
    allergies: List[str]        # each item must be a string
    contact: Dict[str, str]     # keys and values must be strings

In [ ]:
patient_info = {
    "name": "John Doe",
    "age": 30,
    "married": True,
    "allergies": ["penicillin", "peanuts"],
    "contact": {"phone": "123-456-7890", "emergency": "Jane Doe"}
}

patient = Patient(**patient_info)
print(patient)

name='John Doe' age=30 married=True allergies=['penicillin', 'peanuts'] contact={'phone': '123-456-7890', 'emergency': 'Jane Doe'}


<a id='optional-types'></a>

#### **2.1.2 Optional Types**

---

**Important:** Every Pydantic field is **required by default**. Leave one out and you'll get a `ValidationError`.

To make a field truly optional, wrap its type in `Optional` and give it a default of `None`:

In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str
    age: int
    married: bool
    allergies: Optional[List[str]] = None  # skippable — patient may not disclose
    contact: Dict[str, str]

<a id='defaults'></a>

#### **2.1.3 Defaults**

---

Not every optional field should default to `None`. Sometimes you want a real fallback value. Just assign it directly on the field:

In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Optional

class Patient(BaseModel):
    name: str
    age: int
    married: bool
    allergies: Optional[List[str]] = None
    contact: Dict[str, str]
    blood_type: str = "Unknown"   # defaults to "Unknown" if not provided
    visits: int = 0               # defaults to 0

In [ ]:
patient_info = {"name": "John Doe", "age": 30, "married": True, "contact": {"phone": "123-456-7890"}}
patient = Patient(**patient_info)
print(patient.blood_type)  # Unknown
print(patient.visits)      # 0

Unknown
0


<a id='sec-2-2'></a>

### **2.2 Data Validator**

---

Type validator only checks *what type* a value is. But what if you need an `int` that's *between 0 and 120*? That's where data validators come in. They check the actual value, not just the type.

---

<a id='builtin-validators'></a>

#### **2.2.1 Built-in Validators: `EmailStr` and `AnyUrl`**

---

For emails and URLs, don't write a regex yourself. Pydantic has built-in types for these. Just use them instead of `str`:
- **`EmailStr`**: checks valid email format
- **`AnyUrl`**: checks that the URL has a valid scheme (`http://`, `https://`, etc.)

**Note:** `EmailStr` requires `!pip install 'pydantic[email]'`

In [ ]:
!pip install 'pydantic[email]' -q

zsh:1: command not found: pip


In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl

class User(BaseModel):
    name: str
    email: EmailStr   # rejects "hello", accepts "hello@gmail.com"
    profile: AnyUrl   # rejects "my page", accepts "https://github.com/john"

In [ ]:
# Valid
user = User(name="John Doe", email="john@example.com", profile="https://github.com/john")
print(user)

name='John Doe' email='john@example.com' profile=AnyUrl('https://github.com/john')


<a id='field-constraints'></a>

#### **2.2.2 Custom Constraints with `Field()`**

---

Think of it this way: the type annotation says *what kind* of value, and `Field()` says *which specific values are allowed*.

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str   = Field(min_length=2, max_length=50)   # "J" → rejected
    age: int    = Field(ge=0, le=120)                   # 200 → rejected
    visits: int = Field(default=0, ge=0)                # optional, defaults to 0

In [ ]:
try:
    user = User(name="J", age=200)  # name too short, age out of range
except ValidationError as e:
    print(e)

2 validation errors for User
name
  String should have at least 2 characters [type=string_too_short, input_value='J', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
age
  Input should be less than or equal to 120 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


<a id='annotated'></a>

#### **2.2.3 `Annotated`: Write the Rule Once, Use It Everywhere**

---

Here's a problem you'll hit as your codebase grows. Two models both need an age field with the same constraints, so you copy-paste:

```python
class User(BaseModel):
    age: int = Field(ge=0, le=120)   # defined here

class Admin(BaseModel):
    age: int = Field(ge=0, le=120)   # copy-pasted, easy to get out of sync
```

If the rule changes, you have to find and update every copy. `Annotated` fixes this. It lets you attach constraints *directly to the type*, then use that type anywhere:

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field

Age = Annotated[int, Field(ge=0, le=120)]

In [ ]:
class User(BaseModel):
    name: str
    age: Age   # constraints live inside Age — no need to repeat Field()

class Admin(BaseModel):
    age: Age   # same rule, zero duplication

Change it in one place and every model using `Age` picks up the update.

> **Bottom line:** `Annotated` lets you give a type a name with its rules baked in. Define once, use everywhere.

##### **i. Attach descriptions for API docs**

`Annotated` can carry more than just `Field()`. Add a `description` and it shows up automatically in FastAPI/OpenAPI docs with no extra work needed:

In [ ]:
from typing import Annotated
from pydantic import Field

Age      = Annotated[int, Field(ge=0, le=120, description="User age in years")]
Username = Annotated[str, Field(min_length=3, max_length=20, description="Unique username")]

Constraint + description, bundled in one place. Both flow through automatically wherever the type is used.

##### **ii. Build a library of named types**

Take it further. Define all your constrained types at the top of the file. Your model fields become self-explanatory just from their names:

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field, EmailStr

UserId   = Annotated[int, Field(gt=0)]
Username = Annotated[str, Field(min_length=3, max_length=20)]
Email    = Annotated[EmailStr, Field(default="unknown@example.com")]

class User(BaseModel):
    id: UserId
    username: Username
    email: Email

In [ ]:
user = User(id=1, username="johndoe")
print(user)  # email defaults to "unknown@example.com"

id=1 username='johndoe' email='unknown@example.com'


No constraints scattered inside the model. No guessing what values are valid. All the rules are defined in one place and reused.

## **3. Field Validator**

---

`Field()` and `Annotated` let you set rules like *age must be between 0 and 120*. But some rules can't be written as a simple constraint, like *only accept emails from company domains*.

That's what `@field_validator` is for. You write your own validation logic in a method, and Pydantic runs it automatically.

#### **3.1 Validation**

---

Decorate a method with `@field_validator('field_name')`. If the value is wrong, raise a `ValueError`. Pydantic turns it into a `ValidationError`.

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool

    @field_validator('email')
    @classmethod
    def check_email_domain(cls, value) -> str:
        valid_domains = {'hdfc.com', 'kpmg.com'}
        domain = value.split('@')[-1]  # 'kamesh@gmail.com' → 'gmail.com'

        if domain not in valid_domains:
            raise ValueError('Not a valid domain')

        return value

In [ ]:
# gmail.com is not in the allowed list
patient_info = {"name": "Kamesh", "email": "kamesh@gmail.com", "age": 25, "weight": 90.5, "married": True}
try:
    patient = Patient(**patient_info)
except Exception as e:
    print(e)

In [ ]:
# kpmg.com is allowed
patient_info = {"name": "Kamesh", "email": "kamesh@kpmg.com", "age": 25, "weight": 90.5, "married": True}
patient = Patient(**patient_info)
print(patient)

#### **3.2 Transformation**

---

`@field_validator` can also **transform** a value, not just reject it. Whatever you `return` is what Pydantic stores on the model.

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool

    @field_validator('name')
    @classmethod
    def transform_name(cls, value) -> str:
        return value.upper()  # 'kamesh' → 'KAMESH'

In [ ]:
patient_info = {"name": "kamesh", "email": "kamesh@kpmg.com", "age": 25, "weight": 90.5, "married": True}
patient = Patient(**patient_info)
print(patient)  # name is stored as 'KAMESH'

#### **3.3 Before and After Modes**

---

By default, your validator runs **after** Pydantic corrects the type. You can change this with `mode='before'`. Your method then runs on the raw input, before Pydantic touches it.

| Mode | Your method runs | You receive |
|---|---|---|
| `after` (default) | after type correction | already the correct type |
| `before` | before type correction | raw input, whatever the user passed |

**Why does this matter?**

With `mode='after'` (default), Pydantic has already converted `'25'` to `25` by the time your method runs. You always get a clean integer. With `mode='before'`, you get whatever the user typed. If they typed a string and you compare it to a number, Python crashes.

**The catch with `mode='before'`**

A patient passes `age='25'`, a string. In `mode='before'`, your method gets `'25'`, not `25`. Comparing a string to a number crashes:

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool

    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value) -> int:
        if 0 < value < 120: # value is '25' (str) — can't compare with int
            return value
        raise ValueError("Age must be between 0 and 120")

In [ ]:
patient_info = {"name": "Kamesh", "email": "kamesh@kpmg.com", "age": "25", "weight": 90.5, "married": True}
try:
    patient = Patient(**patient_info)
except Exception as e:
    print(e)

Fix: convert to `int` yourself before comparing:

In [ ]:
from pydantic import BaseModel, EmailStr, field_validator

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool

    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value) -> int:
        value = int(value)   # '25' → 25
        if 0 < value < 120:
            return value
        raise ValueError("Age must be between 0 and 120")

In [ ]:
# age='25' now works
patient_info = {"name": "Kamesh", "email": "kamesh@kpmg.com", "age": "25", "weight": 90.5, "married": True}
patient = Patient(**patient_info)
print(patient)

## **4. Model Validator**

---

`@field_validator` works on a single field. But what if your rule involves **multiple fields together**?

For example: if a patient is older than 60, they must have an emergency contact. You can't check this on `age` or `contact_details` alone. You need both at the same time.

That's what `@model_validator` is for. It runs on the entire model after all fields are validated.

#### **4.1 Validation**

---

Use `mode='after'` so all fields are already validated by the time your method runs. Access them via `self`. Return `self` at the end.

In [ ]:
from pydantic import BaseModel, EmailStr, model_validator
from typing import List, Dict
from typing_extensions import Self

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @model_validator(mode='after')
    def validate_emergency_contact(self) -> Self:
        if self.age > 60 and 'emergency' not in self.contact_details:
            raise ValueError("Patients older than 60 must have an emergency contact")
        return self

In [ ]:
# age > 60 but no emergency key in contact_details
patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 65,
    'weight': 80.0,
    'married': True,
    'allergies': ['penicillin'],
    'contact_details': {'phone': '123-456-7890'}  # no 'emergency' key
}

try:
    patient = Patient(**patient_info)
except Exception as e:
    print(e)

1 validation error for Patient
  Value error, Patients older than 60 must have an emergency contact [type=value_error, input_value={'name': 'Kamesh', 'email...phone': '123-456-7890'}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [ ]:
# age > 60 with emergency contact — passes
patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 65,
    'weight': 80.0,
    'married': True,
    'allergies': ['penicillin'],
    'contact_details': {'phone': '123-456-7890', 'emergency': 'Jane Doe'}
}

patient = Patient(**patient_info)
print(patient)

name='Kamesh' email='kamesh@kpmg.com' age=65 weight=80.0 married=True allergies=['penicillin'] contact_details={'phone': '123-456-7890', 'emergency': 'Jane Doe'}


## **5. Computed Field**

---

Sometimes a value doesn't need to be passed in because it can always be calculated from other fields. BMI is a good example. You already have `weight` and `height`. There's no reason to ask the caller to also pass `bmi`.

`@computed_field` lets you define that calculation inside the model. Pydantic runs it automatically and treats the result like any other field.

#### **5.1 Without Computed Field**

---

If you don't use `@computed_field`, you have to calculate BMI outside the model and pass it in. The problem is Pydantic has no idea `bmi` is derived from `weight` and `height`, so it can't catch a wrong value:

In [ ]:
from pydantic import BaseModel, EmailStr
from typing import List, Dict

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]
    bmi: float              # caller must calculate and pass this in

In [ ]:
patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 65,
    'weight': 80.0,
    'height': 1.72,
    'married': True,
    'allergies': ['penicillin'],
    'contact_details': {'phone': '123-456-7890', 'emergency': 'Jane Doe'},
    'bmi': 999.0            # wrong value — Pydantic can't catch this
}

patient = Patient(**patient_info)
print(patient)

name='Kamesh' email='kamesh@kpmg.com' age=65 weight=80.0 height=1.72 married=True allergies=['penicillin'] contact_details={'phone': '123-456-7890', 'emergency': 'Jane Doe'} bmi=999.0


#### **5.2 With Computed Field**

---

Decorate a `@property` with `@computed_field`. Pydantic calls it automatically after the model is created. The caller doesn't pass `bmi` at all. Pydantic derives it from `weight` and `height` every time:

In [ ]:
from pydantic import BaseModel, EmailStr, computed_field
from typing import List, Dict

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    weight: float
    height: float
    married: bool
    allergies: List[str]
    contact_details: Dict[str, str]

    @computed_field                 # Pydantic treats this as a model field
    @property
    def bmi(self) -> float:
        return round(self.weight / (self.height ** 2), 2)

In [ ]:
patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 65,
    'weight': 80.0,
    'height': 1.72,
    'married': True,
    'allergies': ['penicillin'],
    'contact_details': {'phone': '123-456-7890', 'emergency': 'Jane Doe'}
    # no bmi — Pydantic computes it
}

patient = Patient(**patient_info)
print(patient)
print("BMI:", patient.bmi)

name='Kamesh' email='kamesh@kpmg.com' age=65 weight=80.0 height=1.72 married=True allergies=['penicillin'] contact_details={'phone': '123-456-7890', 'emergency': 'Jane Doe'} bmi=27.04
BMI: 27.04


`bmi` is no longer the caller's responsibility. It's always derived correctly from `weight` and `height`. The caller can't accidentally pass a wrong value.

## **6. Nested Models**

---

So far, all fields have been simple types like `str`, `int`, or `List[str]`. But in real data, some fields have *structure of their own*.

For example, an address has a city, a state, and a pin code. If you store it as a plain string, you can't validate the parts individually or access them cleanly.

Nested models let you use one Pydantic model as a field inside another. Each model validates its own fields.

#### **6.1 The Problem with a Flat String**

---

If `address` is just a `str`, you can store anything in it. You can't validate the pin code, and you can't access city or state without splitting the string yourself:

In [ ]:
from pydantic import BaseModel

class Patient(BaseModel):
    name: str
    age: int
    address: str            # city, state, pin all in one string

patient_info = {
    'name': 'Kamesh',
    'age': 26,
    'address': 'Mumbai, Maharashtra, 400076'
}

patient = Patient(**patient_info)
print(patient)
print(patient.address.split(',')[2].strip())   # pin — have to parse it manually

name='Kamesh' age=26 address='Mumbai, Maharashtra, 400076'
400076


It runs, but it's fragile. The format `'Mumbai, Maharashtra, 400076'` is just a convention. If someone passes `'Mumbai-Maharashtra-400076'`, the split breaks silently. There's no way to validate the pin code or make the city field required.

#### **6.2 Using a Nested Model**

---

Define a separate `Address` model with its own fields. Use it as the type for `address` in `Patient`. Now each part of the address is a named, validated field. If the pin code is missing or the city is wrong, you get a clear error pointing to exactly that field:

In [ ]:
from pydantic import BaseModel, EmailStr
from typing import List, Dict

class Address(BaseModel):
    city: str
    state: str
    pin: str                # can add Field(pattern=r'^\d{6}$') to validate 6-digit pin

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    married: bool
    address: Address        # nested model as a field type

In [ ]:
# Way 1 — instantiate Address separately, then pass the object
address_info = {
    'city': 'Mumbai',
    'state': 'Maharashtra',
    'pin': '400076'
}
address = Address(**address_info)   # Address object created first

patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 26,
    'married': False,
    'address': address              # pass the Address object
}

patient = Patient(**patient_info)
print(patient)
print("City:", patient.address.city)

name='Kamesh' email='kamesh@kpmg.com' age=26 married=False address=Address(city='Mumbai', state='Maharashtra', pin='400076')
City: Mumbai


In [ ]:
# Way 2 — keep address_info separate, nest it inside patient_info
address_info = {
    'city': 'Mumbai',
    'state': 'Maharashtra',
    'pin': '400076'
}

patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 26,
    'married': False,
    'address': address_info         # address_info dict nested inside patient_info
}

patient = Patient(**patient_info)
print(patient)
print("City:", patient.address.city)

name='Kamesh' email='kamesh@kpmg.com' age=26 married=False address=Address(city='Mumbai', state='Maharashtra', pin='400076')
City: Mumbai


You don't need to instantiate `Address` separately. Pass the dict directly and Pydantic builds the nested model for you. If any field inside `Address` fails validation, you get a `ValidationError` pointing exactly to which nested field failed.

**Benefits of Nested Models:**

- **Readability**: `patient.address.city` is cleaner than parsing a flat string every time
- **Reusability**: define `Address` once and use it in `Patient`, `Doctor`, `Hospital`, anywhere
- **Validation**: each nested model validates its own fields. Wrong pin code fails at `Address`, not buried in `Patient`
- **Error clarity**: `ValidationError` tells you exactly which field inside which model failed

## **7. Serialization**

---

You have validated data in a Pydantic model. Now you want to save it to a database, or send it as an API response, or write it to a file. None of these accept a Pydantic object. They need a `dict` or a JSON string.

Pydantic has two methods for this:

- `model_dump()` gives you a Python `dict`
- `model_dump_json()` gives you a JSON string

In [ ]:
from pydantic import BaseModel, EmailStr
from typing import List, Dict

class Address(BaseModel):
    city: str
    state: str
    pin: str

class Patient(BaseModel):
    name: str
    email: EmailStr
    age: int
    married: bool
    address: Address

In [ ]:
address_info = {
    'city': 'Mumbai',
    'state': 'Maharashtra',
    'pin': '400076'
}

patient_info = {
    'name': 'Kamesh',
    'email': 'kamesh@kpmg.com',
    'age': 26,
    'married': False,
    'address': address_info
}

patient = Patient(**patient_info)
print(patient)

name='Kamesh' email='kamesh@kpmg.com' age=26 married=False address=Address(city='Mumbai', state='Maharashtra', pin='400076')


#### **7.1 `model_dump()`**

---

Call `model_dump()` on your model to get a plain Python `dict`. You need this when saving to a database, passing data to a function that expects a dict, or logging:

In [ ]:
data = patient.model_dump()
print(type(data))   # plain Python dict
data

<class 'dict'>


{'name': 'Kamesh',
 'email': 'kamesh@kpmg.com',
 'age': 26,
 'married': False,
 'address': {'city': 'Mumbai', 'state': 'Maharashtra', 'pin': '400076'}}

The nested `Address` came out as a dict too. Pydantic handles that automatically.

#### **7.2 `model_dump_json()`**

---

Converts your model directly to a JSON string. Use this when you are sending data over HTTP or writing to a file:

In [ ]:
data = patient.model_dump_json()
print(type(data))   # string, not dict
data

<class 'str'>


'{"name":"Kamesh","email":"kamesh@kpmg.com","age":26,"married":false,"address":{"city":"Mumbai","state":"Maharashtra","pin":"400076"}}'

`True` became `false` in the output. Python uses `True/False`, JSON uses `true/false`. Pydantic converts this for you.

#### **7.3 include and exclude**

---

Sometimes you don't want all fields in the output. Use `include` to get only the fields you need, or `exclude` to drop specific fields:

In [ ]:
# only return name and age
data = patient.model_dump(include={'name', 'age'})
data

{'name': 'Kamesh', 'age': 26}

In [ ]:
# return everything except name and age
data = patient.model_dump(exclude={'name', 'age'})
data

{'email': 'kamesh@kpmg.com',
 'married': False,
 'address': {'city': 'Mumbai', 'state': 'Maharashtra', 'pin': '400076'}}

`include` gives you only the fields you list. `exclude` gives you everything except the fields you list. Both work with `model_dump_json()` as well.